<a href="https://colab.research.google.com/github/KalebKei/thesis/blob/main/Training/trainCIFAR10DVSnote.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook version of train CIFAR10DVS
### This is intended for use in Google Colab. Requires mounting your drive which already has the CIFAR10DVS dataset downloaded

### Optional setup required for COLAB

In [1]:
!git clone https://github.com/KalebKei/thesis
%cd thesis

fatal: destination path 'thesis' already exists and is not an empty directory.
/content/thesis


In [2]:
from google.colab import drive
drive.mount('/content/drive')
%cd Training

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/thesis/Training


In [3]:
!pip install tonic snntorch aedat --quiet # restart runtime after this runs

## Code time

In [4]:
import sys
import argparse
import os
import tonic
from torch.utils.data import DataLoader, random_split
import torch
import torch.nn as nn
from pathlib import Path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
# yeah the name is def not confusing - trust

# custom funcs
import trainhelpers as th
from Models.snn_baseline import SNNModel_CIFAR
from Models.snn_wavelet import WaveletModel_CIFAR

import Encodings.cifarencodings as encodings

In [5]:
debug = False
plot = False
gpu = True
epochs = 50
batch_size = 16
encoding_val = 0
encoding = ""
checkpoint_file = ""
model_filename = "20260728_042901_checkpoint_epoch_15.pt"
model_type = "20260728_042901_hist_checkpoint_epoch_15.pt"
model_type_val = 0
checkpoint_dir = "/content/drive/MyDrive/ModelCheckpoints/CIFAR10DVS"

In [6]:
# Encodings
if(encoding_val > 4):
    sys.exit(f"Error, incorrect encoding type: {encoding_val}")
if(encoding_val == 0):
    transform = encodings.spiketrain_transform
    encoding = "spike_train"
    checkpoint_file = "SpikeTrain"
elif(encoding_val == 1):
    transform = encodings.voxel_grids_transform
    encoding = "voxel_grid"
    checkpoint_file = "VoxelGrids"
elif(encoding_val == 2):
    transform = encodings.dct_transform
    encoding = "dct"
    checkpoint_file = "DCT"
elif(encoding_val == 3):
    transform = encodings.truncated_dct_transform
    encoding = "trunc_dct"
    checkpoint_file = "TruncatedDCT"
elif(encoding_val == 4):
    transform = encodings.aggressive_dct_transform
    encoding = "aggr_dct"
    checkpoint_file = "AggressiveDCT"

# Model loading
if(model_type_val == 0):
    model = SNNModel_CIFAR()
    model_type = "SNN"
elif(model_type_val == 1):
    model = WaveletModel_CIFAR()
    model_type = "FrontEndWaveletSNN"

# Model training continuation
if(model_filename != ""):
    file_path = Path(model_filename)
    if not file_path.is_file():
        sys.exit(f"Model file path {model_filename} does not exist.")
    hist_file_path = Path(model_filename)
    if not hist_file_path.is_file():
        sys.exit(f"Model history file path {args.model_hist_filename} does not exist.")

    checkpoint = torch.load(model_filename, map_location=torch.device('cpu'), weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    history = th.load_hist(args.model_hist_filename)
else:
    history = None

# GPU
if(gpu == True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if(debug):
        print(f"Current training device: {device}")
    model = model.to(device)

In [7]:
import os
import shutil
# 1. Define paths
drive_path = "/content/drive/MyDrive/Datasets/CIFAR10DVS"
local_path = "/content/Datasets/"
dataset="CIFAR10DVS"
full_local = f"{local_path}{dataset}"

# 2. Copy files from Google Drive to local environment if not already copied
if not os.path.exists(f"{local_path}{dataset}"):
    print("Copying dataset from Google Drive to local runtime...")
    shutil.copytree(drive_path, f"{local_path}{dataset}")
    print("Copy complete!")

In [8]:
# Load the dataset and encode
# Translate to frame or whatever
raw_dataset = tonic.datasets.CIFAR10DVS(
    save_to=local_path,
    transform=transform
)

train_size = int(0.8 * len(raw_dataset))
test_size = len(raw_dataset) - train_size

# 3. Randomly split the dataset
train_dataset, test_dataset = random_split(raw_dataset, [train_size, test_size])


# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [9]:
# Test
debug=True
if(debug):
    print(f"Model {model_type} architecture:\n", model, "\n")
    print(f"Testing one forward pass of {model_type} model")
    frames, labels = next(iter(train_loader))

    print(f"Input: {frames.shape}")
    frames = frames.float().to(device)

    output, spikes_count = model(frames)

    print("Successful pass")
    print(f"\tOutput: {output.shape}")
    print("\tFirst layer firing rate:", spikes_count["layer1fr"].item()*100, '%')
    print("\tSecond layer firing rate:", spikes_count["layer2fr"].item()*100, '%')
    if(model_type == "FrontEndWaveletSNN"):
        print("\tThird layer firing rate:", spikes_count["layer3fr"].item()*100, '%')
    print("\tOutput layer firing rate:", spikes_count["outputfr"].item()*100, '%')
    loss_fun = nn.CrossEntropyLoss() # #nofun

    th.validate(model, test_loader, loss_fun, model_type=model_type, device=device)
debug=False



Model SNN architecture:
 SNNModel_CIFAR(
  (conv1): Conv2d(2, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (lif1): Leaky()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (lif2): Leaky()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (lif3): Leaky()
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (avgpool): AdaptiveAvgPool2d(output_size=(1, 1))
  (fc): Linear(in_features=64, out_features=10, bias=True)
  (lif_out): Leaky()
) 

Testing one forward pass of SNN model
Input: torch.Size([16, 50, 2, 128, 128])
Successful pass
	Output: torch.Size([50, 16, 10])
	First layer firing rate: 4.588432237505913 %
	Second layer firing rate: 1.3677701354026794 %
	Output layer firing rate: 0.23750001564621925 %


In [ ]:
# Now time for some train time

loss_fun = nn.CrossEntropyLoss() # #nofun

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4 # i never know what to put this guy at
)

history = th.train(model=model, train_loader=train_loader, test_loader=test_loader, optimizer=optimizer, loss_fun=loss_fun, epochs=epochs, device=device, checkpoint_dir=f"{checkpoint_dir}/{model_type}/{checkpoint_file}", encoding=encoding, model_type=model_type, history=history, debug=debug)

if(plot):
    th.plot_hist(history=history, epochs=epochs)


Epoch 1: 100%|██████████| 500/500 [10:00<00:00,  1.12s/it]
                                                          
Training (50 epochs):   2%|▏         | 1/50 [12:00<9:48:34, 720.71s/it]

Saving history to /content/drive/MyDrive/ModelCheckpoints/CIFAR10DVS/SNN/SpikeTrain/20260728_013626_hist_checkpoint_epoch_1.pt



Epoch 2: 100%|██████████| 500/500 [10:04<00:00,  1.15s/it]
                                                          
Training (50 epochs):   4%|▍         | 2/50 [24:05<9:38:37, 723.27s/it]

Saving history to /content/drive/MyDrive/ModelCheckpoints/CIFAR10DVS/SNN/SpikeTrain/20260728_014831_hist_checkpoint_epoch_2.pt



Epoch 3: 100%|██████████| 500/500 [10:08<00:00,  1.31s/it]
                                                          
Training (50 epochs):   6%|▌         | 3/50 [36:14<9:28:36, 725.88s/it]

...


Epoch 14: 100%|██████████| 500/500 [10:18<00:00,  1.13s/it]
                                                           
Training (50 epochs):  28%|██▊       | 14/50 [2:51:51<7:25:13, 742.04s/it]

Saving history to /content/drive/MyDrive/ModelCheckpoints/CIFAR10DVS/SNN/SpikeTrain/20260728_041617_hist_checkpoint_epoch_14.pt



Epoch 15: 100%|██████████| 500/500 [10:37<00:00,  1.18s/it]
                                                           
Training (50 epochs):  30%|███       | 15/50 [3:04:36<7:16:45, 748.72s/it]

Saving history to /content/drive/MyDrive/ModelCheckpoints/CIFAR10DVS/SNN/SpikeTrain/20260728_042901_hist_checkpoint_epoch_15.pt



Epoch 16:  61%|██████    | 305/500 [06:23<03:56,  1.21s/it]